# Project 3: Build the Eval Dataset
**Sammi Bronow**

CF model is the best-tuned config from Project 2: item-based KNN with Pearson baseline similarity and k=20 neighbors.</small>

### 1. Load the data

In [6]:
import pandas as pd
from surprise import KNNBasic, Dataset, Reader

books   = pd.read_csv("Books.csv",   encoding="latin-1", on_bad_lines="skip")
ratings = pd.read_csv("Ratings.csv", encoding="latin-1", on_bad_lines="skip")

# Use primary author only (first listed), matching app.py
books["primary_author"] = books["authors"].str.split(",").str[0].str.strip()

title_of  = dict(zip(books["book_id"], books["title"]))
author_of = dict(zip(books["book_id"], books["primary_author"]))
avg_rat   = dict(zip(books["book_id"], books["average_rating"]))
year_of   = dict(zip(books["book_id"], books["original_publication_year"]))

### 2. Fit the CF model (pearson_baseline, item-based, k=20)

<small>Item-based collaborative filtering with Pearson baseline similarity and a neighborhood of 20 (best-tuned configuration from Project 2). Filtering for books with at least 10 ratings before training (same methodology as app from Project 2).</small>

In [7]:
book_counts = ratings.groupby("book_id")["rating"].count()
filtered    = ratings[ratings["book_id"].isin(book_counts[book_counts >= 10].index)]

reader   = Reader(rating_scale=(1, 5))
data     = Dataset.load_from_df(filtered[["user_id", "book_id", "rating"]], reader)
trainset = data.build_full_trainset()

model = KNNBasic(
    k=20, min_k=2,
    sim_options={"name": "pearson_baseline", "user_based": False},
    verbose=False
)
model.fit(trainset)
print("Model trained.")

Model trained.


### 3. Select 10 active users and generate top-10 candidates each

<small>We pick users who have rated at least 20 books so that CF has enough signal to produce meaningful candidates. Books with fewer than 20 ratings are excluded from the candidate pool to avoid over-ranking titles with thin evidence — matching the app's popularity filter. The CF predicted score is used only for ranking and is not passed to the re-ranker.</small>

In [8]:
MIN_USER_RATINGS = 20
MIN_BOOK_RATINGS = 20
N_USERS          = 10
N_CANDIDATES     = 10

# Users with enough rating history
active_users = (
    ratings.groupby("user_id")["rating"]
    .count()
    .loc[lambda s: s >= MIN_USER_RATINGS]
    .sample(N_USERS, random_state=6604)
    .index.tolist()
)

# Popular books (enough ratings for reliable CF signal)
popular_books = set(book_counts[book_counts >= MIN_BOOK_RATINGS].index)

def top_candidates(user_id, n=N_CANDIDATES):
    seen = set(ratings.loc[ratings["user_id"] == user_id, "book_id"])
    scored = [
        {
            "title":        title_of.get(bid, "Unknown"),
            "author":       author_of.get(bid, "Unknown"),
            "avg_rating":   avg_rat.get(bid, "N/A"),
            "year":         int(year_of[bid]) if pd.notna(year_of.get(bid)) else "N/A",
            "cf_predicted": model.predict(user_id, bid).est,
        }
        for bid in popular_books - seen
    ]
    scored.sort(key=lambda r: -r["cf_predicted"])
    return scored[:n]

candidates_by_user = {u: top_candidates(u) for u in active_users}

# Preview first user
print(f"Selected users: {active_users}\n")
print(f"Top {N_CANDIDATES} candidates for user {active_users[0]}:")
for c in candidates_by_user[active_users[0]]:
    print(f"  {c['title']} by {c['author']} ({c['year']}, avg rating {c['avg_rating']})")

Selected users: [20319, 51597, 34193, 11063, 11854, 29037, 27011, 11945, 50103, 49289]

Top 10 candidates for user 20319:
  2010: Odyssey Two (Space Odyssey, #2) by Arthur C. Clarke (1982, avg rating 3.89)
  Mistborn Trilogy Boxed Set (Mistborn, #1-3) by Brandon Sanderson (2009, avg rating 4.55)
  The Rise of Endymion (Hyperion Cantos, #4) by Dan Simmons (1997, avg rating 4.16)
  Heretics of Dune (Dune Chronicles #5) by Frank Herbert (1984, avg rating 3.83)
  Hideaway by Dean Koontz (1992, avg rating 3.82)
  The Blinding Knife (Lightbringer, #2) by Brent Weeks (2012, avg rating 4.45)
  Blood Song (Raven's Shadow, #1) by Anthony  Ryan (2011, avg rating 4.47)
  Midnight Tides (The Malazan Book of the Fallen, #5) by Steven Erikson (2004, avg rating 4.34)
  Shadow Puppets (Ender's Shadow, #3) by Orson Scott Card (2002, avg rating 3.88)
  The Second Summer of the Sisterhood (Sisterhood, #2) by Ann Brashares (2003, avg rating 3.78)


### 4. Write the Braintrust CSV

<small>One row per scenario. We pair each user's candidate list with both user requests, producing 20 rows (10 users x 2 requests). `candidates_text` is the pre-formatted block the re-rank prompt and the judge both drop in, listing each candidate as title, author, year, and average rating, the same fields the Project 2 app passes to Gemini.</small>

In [9]:
# Two contrasting user requests: one mood/tone, one content/character
user_requests = [
    "something dark and scary that I can't put down",
    "something with an intense, fast-paced plot and a shocking ending",
]

def candidates_text(cands):
    # Same format as the Project 2 app's re-ranking prompt
    return "\n".join(
        f'- "{c["title"]}" by {c["author"]} ({c["year"]}, avg rating {c["avg_rating"]})'
        for c in cands
    )

rows = [
    {
        "user_id":         u,
        "user_request":    req,
        "candidates_text": candidates_text(candidates_by_user[u]),
    }
    for u in active_users
    for req in user_requests
]

eval_df = pd.DataFrame(rows)
eval_df.to_csv("Project 3 - Eval Data v2.csv", index=False, encoding="utf-8")

print(f"Wrote {len(eval_df)} rows ({N_USERS} users x {len(user_requests)} requests) to Project 3 - Eval Data v2.csv")
eval_df.head()

Wrote 20 rows (10 users x 2 requests) to Project 3 - Eval Data v2.csv


,user_id,user_request,candidates_text
0,20319,something dark and scary that I can't put down,"- ""2010: Odyssey Two (Space Odyssey, #2)"" by A..."
1,20319,"something with an intense, fast-paced plot and...","- ""2010: Odyssey Two (Space Odyssey, #2)"" by A..."
2,51597,something dark and scary that I can't put down,"- ""If I Stay (If I Stay, #1)"" by Gayle Forman ..."
3,51597,"something with an intense, fast-paced plot and...","- ""If I Stay (If I Stay, #1)"" by Gayle Forman ..."
4,34193,something dark and scary that I can't put down,"- ""The Adventures of Tom Sawyer & Adventures o..."
